# Walk-Forward Validation (Скользящая Валидация)

Этот ноутбук реализует самый строгий метод тестирования в алготрейдинге. Мы имитируем реальную работу бота: он учится на 6 месяцах истории, затем торгует на реальных (незнакомых ему) данных следующие 2 месяца. Затем мы сдвигаем окно, дообучаем бота на свежих данных и торгуем дальше. Это позволяет узнать настоящую доходность алгоритма (Out-of-Sample) без переобучения.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize, VecFrameStack
from stable_baselines3.common.monitor import Monitor

from core.data.data_loader import load_crypto_data
from core.features.feature_generator import FeatureGenerator
from custom_envs.trading_env_v5 import TradingEnvV5
from agents.ppo_agent import create_ppo_agent

## 1. Настройка Данных и Окон Валидации

In [ ]:
df = load_crypto_data(symbol='BTCUSDT', start_date='2022-01-01', interval='4h', source='bybit_futures')

fg = FeatureGenerator()
data_features = fg.transform(df)

hmm_cols = [c for c in data_features.columns if 'hmm_regime' in c]
print(f"Data Features shape: {data_features.shape}")

# Настройки WFV
# Для 4-часовых свечей: 6 свечей в день = 180 свечей в 30-дневном месяце
candles_per_month = 180

train_months = 6
test_months = 2

train_window = train_months * candles_per_month
test_window = test_months * candles_per_month

print(f"Обучающее окно: {train_window} свечей ({train_months} мес)")
print(f"Тестовое окно: {test_window} свечей ({test_months} мес)")

## 2. Инициализация Агента (Continuous Learning)
Вместо создания новой нейросети каждый раз (Вариант А), мы используем подход Continuous Learning (Вариант Б). Агент сохраняет свою память и просто 'дочитывает' новые данные по мере их поступления, как это делали бы вы, переобучая бота раз в 2 месяца в продакшене.

In [ ]:
N_STACK = 10

# Создаем фиктивную среду чисто для инициализации весов PPO
dummy_env = TradingEnvV5(df=data_features.iloc[:100], continuous_action=True, t_max=50)
vec_env_init = VecFrameStack(DummyVecEnv([lambda: Monitor(dummy_env)]), n_stack=N_STACK)
vec_env_init = VecNormalize(vec_env_init, norm_obs=True, norm_reward=True, clip_obs=10., clip_reward=10.)

model = create_ppo_agent(
    vec_env_init, 
    extractor_type="gru", 
    num_hmm_states=len(hmm_cols), 
    n_stack=N_STACK,
    tensorboard_log="./tensorboard_logs_wfv/"
)

## 3. Запуск WFV Цикла

In [ ]:
out_of_sample_pnl_curve = []

steps = range(0, len(data_features) - train_window - test_window + 1, test_window)

# Начальный депозит, который будет перетекать из окна в окно (сложный процент)
current_deposit = 100_000.0

# Для сохранения статистики нормализации между окнами
obs_rms = None

for step in steps:
    print(f"\n{'='*50}")
    print(f"🚀 WFV Шаг: Train [{step} : {step+train_window}], Test [{step+train_window} : {step+train_window+test_window}]")
    
    train_df = data_features.iloc[step : step+train_window].reset_index(drop=True)
    test_df = data_features.iloc[step+train_window : step+train_window+test_window].reset_index(drop=True)
    
    # === ФАЗА ОБУЧЕНИЯ ===
    train_env = TradingEnvV5(df=train_df, continuous_action=True, t_max=200, initial_deposit=100_000.0)
    vec_train = VecFrameStack(DummyVecEnv([lambda: Monitor(train_env)]), n_stack=N_STACK)
    vec_train = VecNormalize(vec_train, norm_obs=True, norm_reward=True, clip_obs=10., clip_reward=10.)
    
    # Переносим статистику нормализации с прошлого шага
    if obs_rms is not None:
        vec_train.obs_rms = obs_rms
        
    model.set_env(vec_train)
    
    print("⏳ Обучение на 6-месячном окне...")
    # 30k шагов достаточно для дообучения (fine-tuning)
    model.learn(total_timesteps=30_000, progress_bar=True, reset_num_timesteps=False)
    
    # Сохраняем статистику нормализации
    obs_rms = vec_train.obs_rms
    
    # === ФАЗА ТЕСТИРОВАНИЯ ===
    # t_max=len(test_df) - мы разрешаем агенту дойти до самого конца тестового окна без сброса
    test_env = TradingEnvV5(df=test_df, continuous_action=True, t_max=len(test_df), initial_deposit=current_deposit)
    vec_test = VecFrameStack(DummyVecEnv([lambda: test_env]), n_stack=N_STACK)
    
    # Важно: training=False, чтобы среда не обновляла нормализатор на тестовых данных
    vec_test = VecNormalize(vec_test, norm_obs=True, norm_reward=False, clip_obs=10., clip_reward=10., training=False)
    vec_test.obs_rms = obs_rms
    
    obs = vec_test.reset()
    done = [False]
    
    print("📉 Инференс (Out-of-Sample) на следующих 2 месяцах...")
    while not done[0]:
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, done, info = vec_test.step(action)
        
        # Сохраняем стоимость портфеля на каждом шаге
        real_env = vec_test.venv.envs[0]
        cur_price = real_env._get_current_price()
        port_val = real_env.get_portfolio_value(cur_price)
        out_of_sample_pnl_curve.append(port_val)
        
    current_deposit = port_val
    print(f"✅ Конец окна. Депозит: ${current_deposit:,.2f}")

## 4. Итоговая Кривая Эквити (Out-of-Sample)
Этот график — самая честная оценка работы алгоритма. Здесь показана торговля **только на тех данных, которые алгоритм никогда не видел во время обучения**.

In [ ]:
plt.figure(figsize=(14, 7))
plt.plot(out_of_sample_pnl_curve, color='#2ca02c', linewidth=2, label='Portfolio Value')
plt.title("Out-of-Sample Equity Curve (Walk-Forward Validation)", fontsize=16)
plt.xlabel("Свечи (4h) вне выборки", fontsize=12)
plt.ylabel("Стоимость Портфеля ($)", fontsize=12)
plt.axhline(100_000, color='red', linestyle='--', alpha=0.7, label='Initial Deposit ($100k)')

plt.grid(True, alpha=0.3)
plt.legend(fontsize=12)
plt.show()

roi = (current_deposit - 100_000) / 100_000 * 100
print(f"\n🔥 Итоговый честный ROI: {roi:.2f}%")
print(f"📊 Бэктест охватил {len(steps) * test_months} месяцев чистой торговли вне выборки.")